In [ ]:
#| default_exp core

# Core WebDAV Archiving Utilities

> This module provides tools to upload structured datasets to a WebDAV server, along with metadata, manifest, license, and usage information. It is designed to ensure reproducibility, immutability, and long-term traceability for research datasets.


---

### `hash_folder`

Compute a short MD5 hash of the contents of a folder.

Used to uniquely identify a dataset by its contents, regardless of file order or timestamps.


### `generate_manifest`
Generate a manifest listing all files in a folder.

The output is a pandas.DataFrame containing:

relative path

file size in bytes

last modified timestamp

This is saved as manifest.csv and archived with the dataset.

### `write_standard_license`
Write a standard MIT LICENSE.txt file.

Used for publicly available datasets when no usage restrictions are required.

### `write_readme`
Generate a README.txt file with human-readable metadata and access instructions.

It includes:

Title

Collection date

Description

Source and responsible party

Contact for access request

### `upload_file_to_webdav`
Upload a single file to a WebDAV URL using requests.put.

Detects and reports overwrite conflicts (HTTP 409) with guidance for resolution.

### `log_upload_to_registry`
Append an entry to upload_registry.csv to track uploaded datasets and their metadata.

Each row includes:

Upload date

Label

Dataset hash

Metadata fields

WebDAV archive URL

In [ ]:
#| export
import os
import json
import hashlib
import requests
from datetime import datetime
import pandas as pd
from pathlib import Path
from requests.auth import HTTPBasicAuth
from getpass import getpass
from time import time

In [ ]:
#| export
def hash_folder(folder_path):    
    "Return a short MD5 hash representing the contents of a folder, based on its files."
    h = hashlib.md5()
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):
            fp = os.path.join(root, file)
            with open(fp, 'rb') as f:
                while chunk := f.read(8192):
                    h.update(chunk)
    return h.hexdigest()[:8]

def generate_manifest(folder_path):
    "Generate a dataframe listing all non-hidden files in the folder, with paths, sizes, and timestamps."
    manifest = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            fp = os.path.join(root, file)
            relative_path = os.path.relpath(fp, folder_path)
            size_bytes = os.path.getsize(fp)
            last_modified = datetime.fromtimestamp(os.path.getmtime(fp)).isoformat()
            manifest.append({
                "relative_path": relative_path,
                "size_bytes": size_bytes,
                "last_modified": last_modified
            })
    return pd.DataFrame(manifest)

def write_standard_license(path, license_type="MIT"):
    "Write a standard open-source MIT LICENSE.txt file at the specified path."
    mit_text = """MIT License

Copyright (c) {}

Permission is hereby granted, free of charge, to any person obtaining a copy
...""".format(datetime.today().year)
    Path(path).write_text(mit_text)

def write_restricted_license(path, contact_email: str, author_name: str, year=None):
    "Write a restrictive LICENSE.txt file forbidding commercial or unauthorised use."
    if year is None:
        year = datetime.today().year

    license_text = f"""Restricted Academic Use License

This dataset is not public. It is shared solely for academic, non-commercial use,
and only with prior written permission from the dataset creators.

Redistribution, publication, or commercial use of this data is strictly prohibited
without explicit authorization.

To request access or inquire about terms of use, please contact:

{author_name} – {contact_email}

Copyright © University of Pisa and Collaborators, {year}
"""
    Path(path).write_text(license_text)


def write_readme(path, metadata: dict, contact_email: str):
    "Write a README.txt file with dataset description and contact info"
    title = metadata.get("title", "Unnamed Dataset")
    date = metadata.get("date_collected", "Unknown Date")
    description = metadata.get("description", "No description provided.")
    source = metadata.get("source", "Unknown source")
    responsible = metadata.get("responsible", "Unknown researcher")

    readme_text = f"""Dataset: {title}
Date: {date}
Collected by: {source}
Archived by: {responsible}

Description:
{description}

Usage:
This dataset is NOT publicly released. Academic use is permitted only with explicit
permission from the owner.

To request access or usage rights, please contact:
{responsible} – {contact_email}

License: See LICENSE.txt
"""
    Path(path).write_text(readme_text)


def upload_file_to_webdav(local_file_path, remote_url, auth):
    "Upload a local file to a WebDAV endpoint. Handles and reports overwrite conflicts (HTTP 409)."
    with open(local_file_path, 'rb') as f:
        response = requests.put(remote_url, data=f, auth=auth)

    if response.status_code == 409:
        print(f"⚠️ File already exists at: {remote_url}")
        print("   → WebDAV refused to overwrite it (HTTP 409 Conflict).")
        print("   💡 To fix this:")
        print("     - Change the dataset label to archive a new version, OR")
        print("     - Modify your dataset so it generates a new content hash.")
        print("     - Avoid reusing the same dataset+label if already uploaded.")
    return response


def log_upload_to_registry(
    registry_path: str,
    dataset_url: str,
    label: str,
    metadata: dict,
    local_hash: str
):
    """
    "Append an entry to a local CSV file to track uploaded datasets and associated metadata."

    Parameters
    ----------
    registry_path : str
        Path to the CSV registry file.
    dataset_url : str
        The full URL where the dataset was uploaded.
    label : str
        The dataset label used for the upload.
    metadata : dict
        Metadata used in the upload (title, source, etc.).
    local_hash : str
        Short hash of the dataset contents.
    """
    from datetime import datetime
    import csv

    fieldnames = [
        "upload_date", "label", "title", "source", "date_collected",
        "responsible", "description", "hash", "url"
    ]

    entry = {
        "upload_date": datetime.today().isoformat(),
        "label": label,
        "title": metadata.get("title", ""),
        "source": metadata.get("source", ""),
        "date_collected": metadata.get("date_collected", ""),
        "responsible": metadata.get("responsible", ""),
        "description": metadata.get("description", ""),
        "hash": local_hash,
        "url": dataset_url
    }

    registry_file = Path(registry_path)
    is_new = not registry_file.exists()

    with open(registry_file, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new:
            writer.writeheader()
        writer.writerow(entry)

    print(f"🗂️ Logged upload to registry: {registry_path}")


def upload_dataset_to_webdav(
    source_folder: str,
    base_webdav_url: str,
    label: str,
    metadata: dict,
    username: str,
    contact_email: str,
    license_type: str = "restricted",
    dry_run: bool = False,
    auth=None
) -> str:
    """
    Upload an entire dataset folder to a WebDAV server, creating a dated and hashed archive folder.

    - Archives metadata, manifest, license, and readme
    - Ensures folder immutability via content hashing
    - Optionally supports dry-run mode to preview actions
    - Tracks progress for each uploaded file
    - Logs successful uploads to a local registry
    """
    if not os.path.exists(source_folder):
        raise FileNotFoundError(f"Source folder does not exist: {source_folder}")
    if not os.path.isdir(source_folder):
        raise NotADirectoryError(f"Source path is not a directory: {source_folder}")
    if not any(Path(source_folder).rglob("*.*")):
        raise ValueError(f"Source folder '{source_folder}' contains no files.")

    if auth is None:
        password = getpass(f"Password for {username}: ")        
        auth = HTTPBasicAuth(username, password)
    
    version_hash = hash_folder(source_folder)
    date_str = datetime.today().strftime("%Y%m%d")
    base_path = f"{base_webdav_url}/RRL/archival_data"
    remote_folder_url = f"{base_path}/{label}_{date_str}_{version_hash}"
    
    print(f"\n📁 Uploading to: {remote_folder_url}")

    if not dry_run:
        r = requests.request("MKCOL", remote_folder_url, auth=auth)
        if r.status_code == 405:
            print(f"ℹ️ Folder already exists: {remote_folder_url}")
        elif r.status_code == 201:
            print(f"📁 Created dataset folder: {remote_folder_url}")
        elif r.status_code not in [200, 201, 405]:
            print(f"❌ Failed to create dataset folder {remote_folder_url} → HTTP {r.status_code}")
            return remote_folder_url
    else:
        print(f"(dry run) would create dataset folder: {remote_folder_url}")

    # all_files = [
    #     p for p in Path(source_folder).rglob("*")
    #     if p.is_file() and not p.name.startswith(".")
    # ]
    def is_not_hidden(path, base_folder):
        try:
            rel_parts = path.relative_to(base_folder).parts
            return all(not part.startswith(".") for part in rel_parts)
        except ValueError:
            return False
    
    all_files = [
        p for p in Path(source_folder).rglob("*")
        if p.is_file() and is_not_hidden(p, Path(source_folder))
    ]

    total_files = len(all_files)

    for i, local_file in enumerate(all_files, 1):
        relative_path = os.path.relpath(local_file, source_folder)
        remote_fp_url = f"{remote_folder_url}/{relative_path}".replace("\\", "/")
        file_size_kb = os.path.getsize(local_file) / 1024
        path_parts = Path(relative_path).parts

        for j in range(1, len(path_parts)):
            folder_url = f"{remote_folder_url}/{'/'.join(path_parts[:j])}".replace("\\", "/")
            if not dry_run:
                r = requests.request("MKCOL", folder_url, auth=auth)
                if r.status_code not in [200, 201, 405]:
                    print(f"⚠️ MKCOL failed: {folder_url} → HTTP {r.status_code}")
            else:
                print(f"(dry run) would create folder: {folder_url}")

        print(f"[{i}/{total_files}] Uploading: {relative_path} ({file_size_kb:.1f} KB)")
        if dry_run:
            print(f"(dry run) would upload to: {remote_fp_url}")
        else:
            t0 = time()
            res = upload_file_to_webdav(str(local_file), remote_fp_url, auth)
            dt = time() - t0
            if res.status_code == 409:
                print(f"⚠️ File already exists: {remote_fp_url}")
                print("   → WebDAV refused to overwrite it (HTTP 409 Conflict).")
            elif res.status_code not in [200, 201, 204]:
                print(f"❌ Upload failed: {res.status_code} {res.reason}")
                print(f"→ {res.text}")
            else:
                print(f"   ✅ Done in {dt:.2f}s")

    metadata["hash"] = version_hash
    metadata["archived_on"] = date_str
    manifest_df = generate_manifest(source_folder)

    tmp_dir = Path("/tmp/webdav_upload")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    metadata_path = tmp_dir / "metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=4))

    manifest_path = tmp_dir / "manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)

    license_path = tmp_dir / "LICENSE.txt"
    readme_path = tmp_dir / "README.txt"

    write_restricted_license(license_path, contact_email, metadata.get("responsible", "Unknown"))
    write_readme(readme_path, metadata, contact_email)

    print("⬆️ Uploading metadata, manifest, license, and README...")
    for local_path, name in [(metadata_path, "metadata.json"),
                             (manifest_path, "manifest.csv"),
                             (license_path, "LICENSE.txt"),
                             (readme_path, "README.txt")]:
        target_url = f"{remote_folder_url}/{name}"
        if dry_run:
            print(f"(dry run) would upload: {name} → {target_url}")
        else:
            res = upload_file_to_webdav(str(local_path), target_url, auth)
            if res.status_code == 409:
                print(f"⚠️ File already exists: {target_url}")
                print("   → WebDAV refused to overwrite it (HTTP 409 Conflict).")
            else:
                print(f"{name} → HTTP {res.status_code}")

    print("\n✅ Upload complete.")

    log_upload_to_registry(
        registry_path="upload_registry.csv",
        dataset_url=remote_folder_url,
        label=label,
        metadata=metadata,
        local_hash=version_hash
    )
    
    return remote_folder_url

In [ ]:
#| export
def upload_registry_to_webdav_USEFUL(
    local_registry_path: str,
    remote_registry_path: str,
    auth: HTTPBasicAuth
):
    "Upload the local registry CSV to a known shared WebDAV location."
    if not os.path.exists(local_registry_path):
        print(f"⚠️ Registry not found at {local_registry_path}")
        return
    with open(local_registry_path, "rb") as f:
        res = requests.put(remote_registry_path, data=f, auth=auth)
    if res.status_code in [200, 201, 204]:
        print(f"📤 Uploaded registry to {remote_registry_path}")
    else:
        print(f"❌ Failed to upload registry: HTTP {res.status_code}")

## Verify upload

In [ ]:
#| export
from getpass import getpass
import os, json, hashlib, requests
from datetime import datetime
import pandas as pd
from pathlib import Path
from requests.auth import HTTPBasicAuth
from time import time
from xml.etree import ElementTree as ET
import csv

In [ ]:
#| export
def verify_uploaded_dataset(
    source_folder: str,
    remote_folder_url: str,
    auth: HTTPBasicAuth,
    metadata: dict,
    base_folder: Path = None,
    strict: bool = False
):
    """
    Compare the local dataset folder with its remote WebDAV version.
    Prints a summary of matches, mismatches, and missing files.
    Optionally raises an error if strict mode is enabled and mismatches are found.

    Parameters
    ----------
    source_folder : str
        Local folder that was uploaded.
    remote_folder_url : str
        Base WebDAV URL where the dataset was uploaded.
    auth : HTTPBasicAuth
        Authentication object for WebDAV.
    metadata : dict
        Metadata dictionary containing dataset info.
    base_folder : Path
        Optional: base folder path for resolving relative paths.
    strict : bool
        If True, raise AssertionError if any files are missing or unexpected.
    """
    from urllib.parse import unquote
    from xml.etree import ElementTree as ET

    if base_folder is None:
        base_folder = Path(source_folder)

    def is_not_hidden(path: Path, base_folder: Path):
        try:
            parts = path.relative_to(base_folder).parts
            return all(not p.startswith(".") for p in parts)
        except ValueError:
            return False

    local_files = {
        str(p.relative_to(base_folder)).replace("\\", "/"): p
        for p in Path(source_folder).rglob("*")
        if p.is_file() and is_not_hidden(p, base_folder)
    }

    print(f"🔍 Verifying {len(local_files)} local files against remote at:")
    print(f"{remote_folder_url}\n")

    headers = {"Depth": "infinity"}
    r = requests.request("PROPFIND", remote_folder_url, auth=auth, headers=headers)
    if r.status_code != 207:
        print(f"❌ Failed to list remote folder contents: HTTP {r.status_code}")
        return

    tree = ET.fromstring(r.content)
    remote_files = set()
    for resp in tree.findall("{DAV:}response"):
        href = resp.find("{DAV:}href")
        if href is not None:
            path = href.text
            if path.endswith("/"):
                continue
            full_prefix = Path(remote_folder_url).parts[-1]
            if full_prefix in path:
                rel_path = path.split(full_prefix, 1)[-1]
                rel_path = rel_path.lstrip("/")
                rel_path = unquote(rel_path).replace("\\", "/")
                remote_files.add(rel_path)

    added_files = {"manifest.csv", "metadata.json", "LICENSE.txt", "README.txt"}
    matched_files = set(local_files.keys()) & remote_files
    missing_files = set(local_files.keys()) - remote_files
    extra_files = remote_files - set(local_files.keys()) - added_files

    print(f"✅ Matched files   : {len(matched_files)}")
    print(f"❌ Missing uploads : {len(missing_files)}")
    print(f"⚠️  Unexpected files on server : {len(extra_files)}")

    if missing_files:
        print("\nMissing:")
        for f in sorted(missing_files):
            print(f" - {f}")

    if extra_files:
        print("\nExtra:")
        for f in sorted(extra_files):
            print(f" - {f}")

    if strict and (missing_files or extra_files):
        raise AssertionError("❌ Strict verification failed: mismatches detected.")

    print("\n📦 Verification complete.")
    return {
        "matched": matched_files,
        "missing": missing_files,
        "extra": extra_files
    }


In [ ]:
#| export
def sync_upload_registry(
    local_registry_path: str,
    remote_registry_url: str,
    auth: HTTPBasicAuth,
    merged_local_backup_path: str = "upload_registry_merged.csv"
):
    """
    Safely merge local and remote upload registries, deduplicate, save locally, and upload back to WebDAV.
    Ensures that the _registry folder is created before upload.

    Parameters
    ----------
    local_registry_path : str
        Path to the local registry CSV file.
    remote_registry_url : str
        URL of the remote registry CSV on WebDAV.
    auth : HTTPBasicAuth
        Authentication object for WebDAV access.
    merged_local_backup_path : str
        Path to save the merged registry locally before uploading.
    """
    import io
    from urllib.parse import urlparse, urlunparse

    if not os.path.exists(local_registry_path):
        print(f"⚠️ Local registry does not exist: {local_registry_path}. You should download it first from the server or run `upload_dataset_to_webdav` to upload a new dataset.")
        return

    local_df = pd.read_csv(local_registry_path)

    # Try to download remote registry
    r = requests.get(remote_registry_url, auth=auth)
    if r.status_code == 200:
        remote_df = pd.read_csv(io.StringIO(r.text))
        print(f"🔄 Remote registry downloaded: {len(remote_df)} entries")
    elif r.status_code == 404:
        print("📄 Remote registry not found — creating new one.")
        remote_df = pd.DataFrame(columns=local_df.columns)
    else:
        print(f"❌ Failed to fetch remote registry: HTTP {r.status_code}")
        return

    # Merge and deduplicate
    if not remote_df.empty:
        remote_df = remote_df[[col for col in remote_df.columns if col in local_df.columns]]
    merged_df = pd.concat([local_df, remote_df], ignore_index=True)
    merged_df.drop_duplicates(subset=["hash", "label"], keep="last", inplace=True)

    # Save merged registry locally
    merged_df.to_csv(merged_local_backup_path, index=False)
    print(f"💾 Merged registry saved locally at: {merged_local_backup_path}")

    # Ensure _registry/ folder exists using robust URL parsing
    parsed = urlparse(remote_registry_url)
    registry_folder_path = str(Path(parsed.path).parent)
    registry_folder_url = urlunparse((parsed.scheme, parsed.netloc, registry_folder_path, "", "", ""))

    r_mkcol = requests.request("MKCOL", registry_folder_url, auth=auth)
    if r_mkcol.status_code not in [200, 201, 405]:
        print(f"❌ Failed to create _registry folder: HTTP {r_mkcol.status_code}")
        return

    # Upload the merged registry
    with open(merged_local_backup_path, "rb") as f:
        res = requests.put(remote_registry_url, data=f, auth=auth)

    if res.status_code in [200, 201, 204]:
        print(f"✅ Merged registry uploaded to: {remote_registry_url}")
    else:
        print(f"❌ Failed to upload merged registry: HTTP {res.status_code}")



# Archive Dataset

This method implements the full archival workflow.

What It Does
- 📤 Uploads your dataset to WebDAV using upload_dataset_to_webdav(...)
- 🧪 Verifies all files are present on the server (optional: verify=True)
- 📑 Logs the dataset metadata and hash to a local registry
- 🔁 Merges and uploads the registry to a shared WebDAV location (optional: sync_registry=True)
- 🔒 Supports dry_run=True for safe testing

In [ ]:
#| export
def archive_dataset(
    source_folder: str,
    base_webdav_url: str,
    label: str,
    metadata: dict,
    username: str,
    contact_email: str,
    registry_path: str = "upload_registry.csv",
    remote_registry_url: str = None,
    merged_local_backup_path: str = "upload_registry_merged.csv",
    verify: bool = True,
    strict: bool = False,
    sync_registry: bool = True,
    dry_run: bool = False
) -> str:
    """
    Archive a dataset by uploading to WebDAV, verifying upload, and syncing registry.

    Parameters
    ----------
    source_folder : str
        Path to the dataset folder.
    base_webdav_url : str
        WebDAV base URL.
    label : str
        Dataset label (used for folder naming).
    metadata : dict
        Metadata describing the dataset.
    username : str
        WebDAV login username.
    contact_email : str
        Responsible person's contact email.
    registry_path : str
        Path to the local CSV registry.
    remote_registry_url : str
        WebDAV URL of the remote registry CSV.
    merged_local_backup_path : str
        Local path to store the merged registry.
    verify : bool
        Whether to verify the upload.
    strict : bool
        Whether to raise an error if verification fails.
    sync_registry : bool
        Whether to sync the registry to WebDAV.
    dry_run : bool
        If True, only print actions without uploading.

    Returns
    -------
    str
        URL of the remote dataset folder.
    """
    from requests.auth import HTTPBasicAuth

    print("🔐 Requesting credentials...")
    password = getpass(f"Password for {username}: ")
    auth = HTTPBasicAuth(username, password)
    
    remote_url = upload_dataset_to_webdav(
        source_folder=source_folder,
        base_webdav_url=base_webdav_url,
        label=label,
        metadata=metadata,
        username=username,
        contact_email=contact_email,
        dry_run=dry_run,
        auth=auth
    )

    print('here')
    if verify and not dry_run:
        verify_uploaded_dataset(
            source_folder=source_folder,
            remote_folder_url=remote_url,
            auth=auth,
            metadata=metadata,
            strict=strict
        )

    if sync_registry and not dry_run and remote_registry_url:
        sync_upload_registry(
            local_registry_path=registry_path,
            remote_registry_url=remote_registry_url,
            auth=auth,
            merged_local_backup_path=merged_local_backup_path
        )

    return remote_url


#### Example Usage

In [ ]:
my_username = "MY_USERNAME"

In [ ]:
archive_dataset(
    source_folder="../data/test_dataset",
    base_webdav_url="https://u-drive.unipi.it/remote.php/webdav",    
    label="2025_test_dataset",
    metadata={
        "title": "Dummy test dataset",
        "source": "Synthetic",
        "date_collected": "2025-07-25",
        "description": "Used for validating upload function.",
        "responsible": "Andrea Munafo"
    },
    username=my_username,
    contact_email="andrea.munafo@unipi.it",
    registry_path="upload_registry.csv",
    remote_registry_url="https://u-drive.unipi.it/remote.php/webdav/RRL/archival_data/_registry/upload_registry.csv",
    merged_local_backup_path="upload_registry_merged.csv",
    verify=True,
    strict=True,
    sync_registry=True,
    dry_run=False
)


# Core function test: upload_dataset_to_webdav

This notebook tests the main function in `dataset_archiver.core` using a small test dataset in `data/test_dataset`.

It does a dry-run upload to the WebDAV server and verifies that:
- The dataset folder is hashed
- Metadata and manifest are generated
- Upload returns a correct URL

**NOTE**: This test requires internet access and valid WebDAV credentials.


### Full Archival Workflow

#### 1. Upload the dataset (main function)

In [ ]:
remote_url = upload_dataset_to_webdav(
    source_folder="../data/test_dataset",
    base_webdav_url="https://u-drive.unipi.it/remote.php/webdav",
    label="2025_test_dataset",
    metadata={
        "title": "Dummy test dataset",
        "source": "Synthetic",
        "date_collected": "2025-07-25",
        "description": "Used for validating upload function.",
        "responsible": "Andrea Munafo"
    },
    username=my_username,
    contact_email="andrea.munafo@unipi.it",
    dry_run=False  # Set to True for a test without uploading
)


This will:

- Upload all valid files
- Create manifest.csv, README.txt, LICENSE.txt, metadata.json
- Save/update upload_registry.csv locally



#### 2. Verify upload integrity

In [ ]:
verify_uploaded_dataset(
    source_folder="../data/test_dataset",
    remote_folder_url=remote_url,
    auth=HTTPBasicAuth(my_username, getpass(f"Password for {my_username}: ")),
    metadata={
        "title": "Dummy test dataset",
        "source": "Synthetic",
        "date_collected": "2025-07-25",
        "description": "Used for validating upload function.",
        "responsible": "Andrea Munafo"
    },
    strict=True
)

#### 3. Synchronise the registry with the server

In [ ]:
sync_upload_registry(
    local_registry_path="upload_registry.csv",
    remote_registry_url="https://u-drive.unipi.it/remote.php/webdav/RRL/archival_data/_registry/upload_registry.csv",
    auth=HTTPBasicAuth(my_username, getpass(f"Password for {my_username}: ")),
    merged_local_backup_path="upload_registry_merged.csv"
)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()